In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.stats import norm
from scipy.optimize import brentq
from scipy.optimize import minimize_scalar

# GCV Contraction

## Data generation functions

In [2]:
def spiked_covariance(D):
    """
    Spiked-type covariance eigenvalues.

    Args:
        D: the number of parameters in the data matrix

    Returns:
        Sigma: the covariance matrix
    """
    x = np.linspace(-3, 3, D)
    eigenvalues = norm.pdf(x)
    eigenvalues = eigenvalues / eigenvalues.max()
    Sigma = np.diag(eigenvalues)
    return Sigma

def generate_X(N, D, Sigma, rng):
    """
    Generates a random mean-centered X from a multivariate normal distribution.

    Args:
        N: the number of observations.
        D: the number of parameters.
        Sigma: the covariance matrix of the data matrix
            X with shape DxD.
        rng: A NumPy random number generator

    Returns:
        A matrix X of the shape NxD (N rows, D columns).
    """
    mu = np.zeros(D)
    X = rng.multivariate_normal(mean=mu, cov=Sigma, size=N)
    return X - X.mean(axis=0)


def generate_Y(X, N, theta, epsilon, rng):
    """
    Generates a random mean-centered y from a standard normal distribution.

    Args:
        X: the data matrix with shape NxD.
        N: the number of observations.
        theta: the true parameter vector.
        epsilon: the noise amplitude.
        rng: A NumPy random number generator

    Returns:
        A column vector y of the shape Nx1 (N rows, 1 column).
    """
    z = (rng.standard_normal(size=N))[:, np.newaxis]
    y = X @ theta + epsilon * z
    return y - y.mean(axis=0)

## GCV functions and stuff

\begin{equation}
    A(\lambda) = \left[
    \displaystyle\sum_{j=1}^m
    \frac{\lambda^2 a_j^2}{n(\sigma_j^2+\lambda)^2}
    +\displaystyle\sum_{j=m+1}^n\frac{a_j^2}{n}
    \right]
    \left[
    \displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2}{(\sigma_j^2+\lambda)^2}
    \right]
\end{equation}

\begin{equation}
    B(\lambda) = \displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2 a_j^2}{(\sigma_j^2+\lambda)^3} \left[1-\dfrac{1}{n}\displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2}{\sigma_j^2+\lambda}\right]
\end{equation}

\begin{equation}
    A'(\lambda) = \left[\frac{2\lambda}{n} \sum_{j = 1}^m \frac{\sigma_j^2 a_j^2}{(\sigma_j^2 + \lambda)^3}
    \right]
    \left[
    \displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2}{(\sigma_j^2+\lambda)^2}
    \right] -2 \left[
    \displaystyle\sum_{j=1}^m
    \frac{\lambda^2 a_j^2}{n(\sigma_j^2+\lambda)^2}
    +\displaystyle\sum_{j=m+1}^n\frac{a_j^2}{n}
    \right]
    \left[
    \displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2}{(\sigma_j^2+\lambda)^3}
    \right]
\end{equation}

\begin{equation}
    B'(\lambda) = -3\left[\displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2 a_j^2}{(\sigma_j^2+\lambda)^4} \right]\left[1-\dfrac{1}{n}\displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2}{\sigma_j^2+\lambda}\right] + \left[\displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2 a_j^2}{(\sigma_j^2+\lambda)^3} \right]\left[\dfrac{1}{n}\displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2}{(\sigma_j^2+\lambda)^2}\right].
\end{equation}

In [3]:
def A(lam, n, a, singular_values):
    a = np.asarray(a, dtype=float)
    s2 = np.asarray(singular_values, dtype=float)**2
    m = len(s2)
    d = s2 + lam

    residual = (
        np.sum(lam**2 * a[:m]**2 / d**2)
        + np.sum(a[m:]**2)
    ) / n

    return residual * np.sum(s2 / d**2)


def B(lam, n, a, singular_values):
    a = np.asarray(a, dtype=float)
    s2 = np.asarray(singular_values, dtype=float)**2
    m = len(s2)
    d = s2 + lam

    weighted_sum = np.sum(s2 * a[:m]**2 / d**3)
    factor = 1 - np.sum(s2 / d) / n

    return weighted_sum * factor


def A_prime(lam, n, a, singular_values):
    a = np.asarray(a, dtype=float)
    s2 = np.asarray(singular_values, dtype=float)**2
    m = len(s2)
    d = s2 + lam

    residual = (
        np.sum(lam**2 * a[:m]**2 / d**2)
        + np.sum(a[m:]**2)
    ) / n
    residual_prime = (2 * lam / n) * np.sum(
        s2 * a[:m]**2 / d**3
    )

    return (
        residual_prime * np.sum(s2 / d**2)
        - 2 * residual * np.sum(s2 / d**3)
    )


def B_prime(lam, n, a, singular_values):
    a = np.asarray(a, dtype=float)
    s2 = np.asarray(singular_values, dtype=float)**2
    m = len(s2)
    d = s2 + lam

    weighted_sum = np.sum(s2 * a[:m]**2 / d**3)
    weighted_sum_prime = -3 * np.sum(s2 * a[:m]**2 / d**4)
    factor = 1 - np.sum(s2 / d) / n
    factor_prime = np.sum(s2 / d**2) / n

    return weighted_sum_prime * factor + weighted_sum * factor_prime

\begin{equation}
    \lambda 
    = \frac{\left[
    \displaystyle\sum_{j=1}^m
    \frac{\lambda^2 a_j^2}{n(\sigma_j^2+\lambda)^2}
    +\displaystyle\sum_{j=m+1}^n\frac{a_j^2}{n}
    \right]
    \left[
    \displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2}{(\sigma_j^2+\lambda)^2}
    \right]}{\displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2 a_j^2}{(\sigma_j^2+\lambda)^3} \left[1-\dfrac{1}{n}\displaystyle\sum_{j=1}^m
    \frac{\sigma_j^2}{\sigma_j^2+\lambda}\right]} = H(\lambda)
\end{equation}

\begin{equation}
    H'(\lambda) = \dfrac{A'(\lambda)}{B(\lambda)} - \dfrac{A(\lambda)B'(\lambda)}{(B(\lambda))^2}
\end{equation}

$$F(\lambda) = \frac{\lambda + H(\lambda)}{2}$$

$$F'(\lambda) = \dfrac{d}{d\lambda} \left[ \frac{\lambda + H(\lambda)}{2} \right] = \frac{1 + H'(\lambda)}{2}$$

\begin{equation}
    F'(\lambda) = \dfrac{1}{2} \left[1 +  \dfrac{A'(\lambda)}{B(\lambda)} - \dfrac{A(\lambda)B'(\lambda)}{(B(\lambda))^2} \right].
\end{equation}



In [4]:
def H(lam, n, a, singular_values):
    A_val = A(lam, n, a, singular_values)
    B_val = B(lam, n, a, singular_values)

    if B_val == 0:
        raise ZeroDivisionError("H is undefined when B(lam) = 0.")

    return A_val / B_val


def H_prime(lam, n, a, singular_values):
    A_val = A(lam, n, a, singular_values)
    B_val = B(lam, n, a, singular_values)

    if B_val == 0:
        raise ZeroDivisionError("H_prime is undefined when B(lam) = 0.")

    Ap_val = A_prime(lam, n, a, singular_values)
    Bp_val = B_prime(lam, n, a, singular_values)

    return Ap_val / B_val - A_val * Bp_val / B_val**2


def F(lam, n, a, singular_values):
    return (lam + H(lam, n, a, singular_values)) / 2


def F_prime(lam, n, a, singular_values):
    return (1 + H_prime(lam, n, a, singular_values)) / 2

## Real testing

### Data generation

In [5]:
rng = np.random.default_rng(seed=77)

EPSILON = 0.1
ASPECT_RATIO = 0.3  # aspect ratio (d/N)
N = 1000 # observations
D = int(ASPECT_RATIO * N)  # number of parameters D

theta = rng.standard_normal(size=D)[:, np.newaxis]
theta = theta / np.linalg.norm(theta)

covariance = spiked_covariance(D)
X = generate_X(N, D, covariance, rng)
y = generate_Y(X, N, theta, EPSILON, rng)

U, sigma, Vt = np.linalg.svd(X)
a = U.T @ y

In [6]:
def G(lam, n, a, singular_values):
    a = np.asarray(a)
    s2 = np.asarray(singular_values) ** 2
    m = len(s2)

    numerator = (
        np.sum(lam**2 * a[:m]**2 / (s2 + lam)**2)
        + np.sum(a[m:]**2)
    ) / n

    denominator = (1 - np.sum(s2 / (s2 + lam)) / n)**2

    return numerator / denominator

result = minimize_scalar(G, args=(N, a, sigma))

optimal_lambda = result.x 
optimal_gcv = result.fun

print("Optimal lambda:", optimal_lambda)
print("Optimal GCV:", optimal_gcv)

Optimal lambda: 0.00034977051686715775
Optimal GCV: 0.0147429547381049


### Level set

In [10]:
def fprime_levelsets(lam_max, steps, n, a, singular_values):
    def g(lam):
        return abs(F_prime(lam, n, a, singular_values)) - 1

    lam_grid = np.linspace(0, lam_max, steps)[1:]
    f_values = np.array([g(lam) for lam in lam_grid])
    roots = []

    for i in range(len(lam_grid) - 1):
        lam_1, lam_2 = lam_grid[i], lam_grid[i+1]
        f_1, f_2 = f_values[i], f_values[i+1]

        if f_1 == 0:
            roots.append(lam_1)

        if f_1 * f_2 < 0:
            root = brentq(g, lam_1, lam_2)
            roots.append(root)

    if f_values[-1] == 0:
        roots.append(lam_grid[-1])

    if not roots and np.all(np.abs(f_values + 1) < 1):
        print("The entire interval is valid.")
        roots.extend([0.0, float(lam_max)])

    return sorted(list(set(roots)))

In [11]:
LAM_MAX = 10 * optimal_lambda
STEPS = 10000
endpoints = fprime_levelsets(LAM_MAX, STEPS, N, a, sigma)
print("Endpoints:", endpoints)

The entire interval is valid.
Endpoints: [0.0, 0.0034977051686715774]


In [13]:
lam_grid = np.linspace(0, LAM_MAX, 10000)[1:]
f_values = np.array([F_prime(lam, N, a, sigma) for lam in lam_grid])

print("Are all the values finite (T/F)?", np.all(np.isfinite(f_values)))
print("Minimum value of F':", np.min(f_values))
print("Maximum value of F':", np.max(f_values))

Are all the values finite (T/F)? True
Minimum value of F': 0.5000268374995797
Maximum value of F': 0.5000526684839776


### Fixed-point iteration

## Data analysis